In [2]:
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
import os
from rag_helper import RAGBase

In [3]:
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course'],
    id_field="doc_id",      # <-- use the existing id so if it's re-run, it will update existing records instead of creating duplicates
    db_path='faq.db'
)

In [4]:
index.search("how do i join a course")

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.',
  'doc_id': 'bd31146b0e'},
 {'id': '85384a18e5',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'OpenAI: Do I have to subscribe and pay for Open AI API for this course?',
  'answer': "No, you don't have to pay for this service in order to complete the course homeworks. You can use free or low-cost alternatives listed in the course GitHub repo.\n\nSee the course list of [OpenAI API alternatives](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/awesome-llms.

In [5]:
openai_client = OpenAI(api_key=os.getenv("OPENROUTER_API_KEY"),
                       base_url="https://openrouter.ai/api/v1"
)

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)




Yes, you can join the course now! However, there are a few important details to note:

1. **Registration is optional**: You don


In [6]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

In [7]:
assistant.rag("How do I get a certificate?")

'\nBased on the FAQ context, here\'s how to get a certificate:\n\n**To earn a certificate, you must:**\n\n1. **Complete the course with a "live" cohort** - You cannot get a certificate in self-paced mode'

In [8]:
search_results = index.search("How do I run ollama locally?")

In [9]:
search_results

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [10]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [11]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [27]:
messages = [{ "role": "user", "content": "how get certified?" }]

In [28]:
response = openai_client.responses.create(
    model="poolside/laguna-m.1:free",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseReasoningItem(id='rs_tmp_6py3g94urwm', summary=[], type='reasoning', content=[Content(text='\nI need to search the FAQ database to find information about certification processes or requirements.\n', type='reasoning_text')], encrypted_content=None, status='completed', format='unknown'),
 ResponseOutputMessage(id='msg_tmp_nbuq7se545c', content=[ResponseOutputText(annotations=[], text='\n\n', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None),
 ResponseFunctionToolCall(arguments='{"query": "How to get certified"}', call_id='chatcmpl-tool-af5d01f04bf34c8d92ae37ba403eed75', name='search', type='function_call', id='fc_tmp_ia8pcwualss', namespace=None, status='completed')]

In [29]:
call = response.output

In [30]:
call

[ResponseReasoningItem(id='rs_tmp_6py3g94urwm', summary=[], type='reasoning', content=[Content(text='\nI need to search the FAQ database to find information about certification processes or requirements.\n', type='reasoning_text')], encrypted_content=None, status='completed', format='unknown'),
 ResponseOutputMessage(id='msg_tmp_nbuq7se545c', content=[ResponseOutputText(annotations=[], text='\n\n', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None),
 ResponseFunctionToolCall(arguments='{"query": "How to get certified"}', call_id='chatcmpl-tool-af5d01f04bf34c8d92ae37ba403eed75', name='search', type='function_call', id='fc_tmp_ia8pcwualss', namespace=None, status='completed')]

In [31]:
import json

call = response.output[2]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [32]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [33]:
messages 

[{'role': 'user', 'content': 'how get certified?'},
 ResponseReasoningItem(id='rs_tmp_6py3g94urwm', summary=[], type='reasoning', content=[Content(text='\nI need to search the FAQ database to find information about certification processes or requirements.\n', type='reasoning_text')], encrypted_content=None, status='completed', format='unknown'),
 ResponseOutputMessage(id='msg_tmp_nbuq7se545c', content=[ResponseOutputText(annotations=[], text='\n\n', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None),
 ResponseFunctionToolCall(arguments='{"query": "How to get certified"}', call_id='chatcmpl-tool-af5d01f04bf34c8d92ae37ba403eed75', name='search', type='function_call', id='fc_tmp_ia8pcwualss', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'chatcmpl-tool-af5d01f04bf34c8d92ae37ba403eed75',
  'output': '[\n  {\n    "id": "9f689c185f",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Q

In [34]:
response = openai_client.responses.create(
    model="poolside/laguna-m.1:free",
    input=messages,
    tools=[search_tool],
)

response.output_text

'\n\n'

In [35]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(1351, 145)

In [36]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176
